# 🔮 PushT Neural Simulator — try it

Drive the DreamerV4 pushT world model from Python with a **Gymnasium-style API**.
Everything you see is *imagined* by the model — give it actions, watch it dream the consequences.

**Before you run**
- Server up on the GPU host:  `python scripts/serve.py`  (default `http://localhost:8077`)
- Client installed:  `pip install ./clients/neuralsim`  (deps: `requests` + `numpy`)
- Run this notebook with the `dreamerv4` kernel (it has `neuralsim`, `matplotlib`, `ipywidgets`).

> ⚠️ **Single environment.** The server hosts *one* live rollout. If the browser page
> (`http://localhost:8077/`) is open it will fight this notebook over the same session —
> use one at a time. If a `step()` ever errors with "no active session", just `reset()` again.

In [ ]:
%matplotlib inline
import time, math
import numpy as np
import matplotlib.pyplot as plt
import neuralsim

SERVER = "http://localhost:8077"      # <- change if your server is elsewhere
env = neuralsim.make(SERVER)
env.info_spec

## 1 · Reset and look at the starting frame
`reset()` seeds the model with a short real clip and returns the first observation — a 256×256 RGB image.

In [ ]:
obs, info = env.reset(seed=0)
print("obs:", obs.shape, obs.dtype, "  info:", info)
plt.figure(figsize=(4, 4)); plt.imshow(obs); plt.axis("off"); plt.title("initial observation"); plt.show()

## 2 · Step with actions
`action = [vertical, horizontal]`, each in `[-1, 1]`.
Browser mapping: **up-arrow → `action[0] = -1`**, **right-arrow → `action[1] = +1`**.

Here we run a slow circle in action space and collect the imagined frames.

> **Reward is always `0.0`** — this checkpoint has no reward head; it's a world-model sandbox,
> not a scored task. Compute any reward you need yourself from `obs`.

In [ ]:
obs, info = env.reset(seed=0)
frames = [obs]
for t in range(60):
    action = np.array([math.sin(t / 12), math.cos(t / 12)], dtype=np.float32)  # circle — replace with your policy
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(obs)
    if terminated or truncated:
        break
print(f"{len(frames) - 1} steps · server compute {info.get('step_ms')} ms/step")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, i in zip(axes.ravel(), np.linspace(0, len(frames) - 1, 10).astype(int)):
    ax.imshow(frames[i]); ax.set_title(f"t={i}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3 · Play it back as an animation

In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(4, 4)); ax.axis("off")
im = ax.imshow(frames[0])
def _update(i):
    im.set_data(frames[i])
    return (im,)
anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=200, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 4 · Live view (paced to 5 Hz)
The model computes a frame in ~25 ms, but pushT was trained at `dt = 0.2 s`, so we pace the
loop to **5 Hz** — one step per 0.2 s — so a held action maps to realistic motion (the same
reason the browser page runs at 5 Hz).

In [ ]:
from IPython.display import clear_output

obs, info = env.reset(seed=1)
target_dt = 1.0 / 5
fig, ax = plt.subplots(figsize=(4, 4))
for t in range(40):
    tic = time.time()
    obs, r, terminated, truncated, info = env.step([-0.8, 0.0])   # push "up"
    ax.clear(); ax.imshow(obs); ax.axis("off"); ax.set_title(f"t={t+1}  ·  push up")
    clear_output(wait=True); display(fig)
    if terminated or truncated:
        break
    time.sleep(max(0.0, target_dt - (time.time() - tic)))
plt.close(fig)

## 5 · Write your own policy
Replace `my_policy` with anything that maps an observation to an action in `[-1, 1]^2`.

In [ ]:
def my_policy(obs):
    # obs: (256, 256, 3) uint8. Return [vertical, horizontal] in [-1, 1].
    return env.action_space.sample()          # <- your policy here

obs, info = env.reset(seed=2)
for t in range(40):
    obs, r, terminated, truncated, info = env.step(my_policy(obs))
    if terminated or truncated:
        break
plt.figure(figsize=(4, 4)); plt.imshow(obs); plt.axis("off")
plt.title(f"after {t+1} policy steps"); plt.show()

## 6 · Latent observations (bandwidth-light)
Ask for the model's latent instead of the decoded image — `(256, 32)` float16. Handy for
latent-space policies; no image decode on either side.

In [ ]:
lat_env = neuralsim.make(SERVER, obs_type="latent")
lobs, _ = lat_env.reset(seed=0)
print("latent obs:", lobs.shape, lobs.dtype)
lobs2, r, term, trunc, info = lat_env.step([0.5, -0.3])
print("after one step:", lobs2.shape, "| mean |Δlatent| =", float(np.abs(lobs2 - lobs).mean()))
lat_env.close()

## 7 · Interactive control (sliders)
Set an action and step. `action[0]`: **−1 up / +1 down**.  `action[1]`: **−1 left / +1 right**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

env.reset(seed=0)
a0 = widgets.FloatSlider(value=-0.8, min=-1, max=1, step=0.05, description="a0 up−/down+")
a1 = widgets.FloatSlider(value=0.0,  min=-1, max=1, step=0.05, description="a1 left−/right+")
go = widgets.Button(description="step ×5", button_style="primary")
out = widgets.Output()

def _step(_):
    obs = None
    for _ in range(5):
        obs, r, term, trunc, info = env.step([a0.value, a1.value])
    with out:
        out.clear_output(wait=True)
        plt.figure(figsize=(3.6, 3.6)); plt.imshow(obs); plt.axis("off"); plt.show()

go.on_click(_step)
display(widgets.HBox([a0, a1]), go, out)

## Done — and driving it by hand

```python
env.close()   # free the session so the browser (or another client) can take over
```

To steer it yourself, open **http://localhost:8077/** in a browser (arrow keys / WASD, or a
gamepad's right stick). Close this notebook's `env` first so they don't fight over the single
environment. Full API + sharing notes: `simserver/README.md`.

In [ ]:
env.close()